# Composer Classification Feature Dataset

## Overview

This dataset contains numerical features extracted from MIDI files from nine classical composers:

- Bach
- Bartók
- Byrd
- Chopin
- Handel
- Hummel
- Mendelssohn
- Mozart
- Schumann

Each row represents one MIDI file. The `composer` column is the target label used for classification.

Generated files:

- data/train_features.csv
- data/dev_features.csv
- data/test_features.csv


## Feature Descriptions

### Metadata Features

composer  
- Composer label (classification target).

filename  
- Original MIDI filename.  
- Used for reference only and should not be included as a model input feature.


## General Musical Features

**tempo**  
- Estimated tempo of the MIDI file in beats per minute (BPM).
- Represents the speed of the musical performance.

**num_notes**  
- Total number of notes in the MIDI file.
- Represents the overall amount of musical activity.

**num_chords**  
- Number of detected chords using `music21` chordification.
- Represents harmonic activity.

**avg_pitch**  
- Average MIDI pitch value of all notes.
- Represents the overall register of the composition.

**pitch_range**  
- Difference between the highest and lowest note.
- Represents the total pitch span of the composition.

**avg_duration**  
- Average note duration in seconds.
- Represents rhythmic characteristics and note length tendencies.

**avg_velocity**  
- Average MIDI velocity value.
- Represents average note intensity/dynamics.

**note_density**  
- Number of notes divided by total MIDI duration.
- Represents how musically dense or active a piece is.


## Pitch Distribution Features

The MIDI pitch classes represent the 12 chromatic notes:

pitch_class_0 = C  
pitch_class_1 = C#/Db  
pitch_class_2 = D  
pitch_class_3 = D#/Eb  
pitch_class_4 = E  
pitch_class_5 = F  
pitch_class_6 = F#/Gb  
pitch_class_7 = G  
pitch_class_8 = G#/Ab  
pitch_class_9 = A  
pitch_class_10 = A#/Bb  
pitch_class_11 = B  

- Normalized histogram of pitch usage.
- Each value represents the proportion of notes belonging to that pitch class.
- Captures tonal and harmonic tendencies of each composition.


## Derived Features

**range_normalized**

- Formula:
  pitch_range / avg_pitch

- Normalizes pitch range relative to the average register.


**notes_per_chord**

- Formula:
  num_notes / num_chords

- Represents the amount of note activity occurring per harmonic event.


**chord_density**

- Formula:
  num_chords / num_notes

- Represents the frequency of harmonic changes relative to note activity.


**velocity_variation**

- Formula:
  avg_velocity / tempo

- Represents the relationship between dynamics and tempo.


**tempo_note_ratio**

- Formula:
  tempo / num_notes

- Represents the relationship between performance speed and note activity.


**chromatic_ratio**

- Percentage of notes belonging to chromatic pitch classes.
- Measures the amount of chromatic pitch usage and harmonic complexity.


**pitch_entropy**

- Measures how evenly distributed the pitch classes are.
- Higher values indicate more varied pitch usage.
- Lower values indicate stronger concentration around certain pitches.


**pitch_class_variance**

- Measures the variance of pitch class usage.
- Higher values indicate stronger differences between frequently and rarely used pitch classes.


## Pipeline

MIDI Files

↓

Feature extraction using `pretty_midi`, Chord analysis using `music21`, Feature calculation, & CSV feature datasets

↓

Machine learning composer classification

In [ ]:
import os
import pandas as pd
import numpy as np

import music21
import pretty_midi
from music21 import converter, chord

In [ ]:
BASE_DIR = "Composer_Dataset/NN_midi_files_extended"

COMPOSERS = [
    "bach",
    "bartok",
    "byrd",
    "chopin",
    "handel",
    "hummel",
    "mendelssohn",
    "mozart",
    "schumann"
]

def extract_features(filepath, composer):
    try:
        midi = pretty_midi.PrettyMIDI(filepath)

        notes = []
        durations = []
        velocities = []

        for instrument in midi.instruments:
            for note in instrument.notes:
                notes.append(note.pitch)
                durations.append(note.end - note.start)
                velocities.append(note.velocity)

        if len(notes) == 0:
            return None

        tempo = midi.estimate_tempo()

        pitch_hist = midi.get_pitch_class_histogram()

        note_density = len(notes) / midi.get_end_time()

        avg_pitch = np.mean(notes)
        pitch_range = np.max(notes) - np.min(notes)
        avg_duration = np.mean(durations)
        avg_velocity = np.mean(velocities)

        ####################################################
        # Count chords using music21
        ####################################################

        score = converter.parse(filepath)

        chords = score.chordify()

        chord_count = 0

        for c in chords.recurse().getElementsByClass(chord.Chord):
            chord_count += 1

        ####################################################

        features = {
            "composer": composer,
            "filename": os.path.basename(filepath),
            "tempo": tempo,
            "num_notes": len(notes),
            "num_chords": chord_count,
            "avg_pitch": avg_pitch,
            "pitch_range": pitch_range,
            "avg_duration": avg_duration,
            "avg_velocity": avg_velocity,
            "note_density": note_density,
        }

        for i in range(12):
            features[f"pitch_class_{i}"] = pitch_hist[i]

        return features

    except Exception as e:
        print(f"Error reading {filepath}")
        print(e)
        return None


def process_dataset(split):

    rows = []

    split_path = os.path.join(BASE_DIR, split)

    for composer in COMPOSERS:

        composer_folder = os.path.join(split_path, composer)

        if not os.path.exists(composer_folder):
            continue

        print(f"Processing {composer}...")

        for file in os.listdir(composer_folder):

            if file.endswith(".mid"):

                filepath = os.path.join(composer_folder, file)

                features = extract_features(filepath, composer)

                if features is not None:
                    rows.append(features)

    df = pd.DataFrame(rows)

    output = f"data/{split}_features.csv"

    df.to_csv(output, index=False)

    print(f"Saved {output}")


for split in ["train", "dev", "test"]:
    process_dataset(split)

In [3]:
def add_features(csv_path):

    df = pd.read_csv(csv_path)

    # ----------------------------
    # Pitch class distribution features
    # ----------------------------

    def pitch_class_entropy(row):

        values = np.array(
            [row[f"pitch_class_{i}"] for i in range(12)]
        )

        values = values / (values.sum() + 1e-10)

        return -np.sum(
            values * np.log(values + 1e-10)
        )


    def pitch_class_variance(row):

        values = np.array(
            [row[f"pitch_class_{i}"] for i in range(12)]
        )

        return np.var(values)


    df["pitch_entropy"] = df.apply(
        pitch_class_entropy,
        axis=1
    )

    df["pitch_class_variance"] = df.apply(
        pitch_class_variance,
        axis=1
    )


    # ----------------------------
    # Existing pitch features
    # ----------------------------

    df["range_normalized"] = (
        df["pitch_range"] /
        (df["avg_pitch"] + 1e-6)
    )


    # ----------------------------
    # Density features
    # ----------------------------

    df["notes_per_chord"] = (
        df["num_notes"] /
        (df["num_chords"] + 1)
    )

    df["chord_density"] = (
        df["num_chords"] /
        (df["num_notes"] + 1)
    )


    # ----------------------------
    # Rhythm approximations
    # ----------------------------

    df["velocity_variation"] = (
        df["avg_velocity"] /
        (df["tempo"] + 1)
    )

    df["tempo_note_ratio"] = (
        df["tempo"] /
        (df["num_notes"] + 1)
    )


    # ----------------------------
    # Chromaticism estimate
    # ----------------------------

    white_keys = [
        0,2,4,5,7,9,11
    ]

    pitch_total = np.zeros(len(df))

    chromatic = np.zeros(len(df))

    for i in range(12):
        pitch_total += df[f"pitch_class_{i}"]

        if i not in white_keys:
            chromatic += df[f"pitch_class_{i}"]

    df["chromatic_ratio"] = (
        chromatic /
        (pitch_total + 1e-6)
    )


    # ----------------------------
    # Pitch entropy
    # ----------------------------

    def entropy(row):

        values = np.array(
            [row[f"pitch_class_{i}"] for i in range(12)]
        )

        values = values / (values.sum()+1e-10)

        return -np.sum(
            values*np.log(values+1e-10)
        )

    df["pitch_entropy"] = df.apply(
        entropy,
        axis=1
    )


    # Save
    df.to_csv(csv_path, index=False)

    print(
        f"Updated {csv_path}: {df.shape[1]} features"
    )


# Update all splits

for split in ["train", "dev", "test"]:

    path = f"data/{split}_features.csv"

    if os.path.exists(path):
        add_features(path)

Updated data/train_features.csv: 30 features
Updated data/dev_features.csv: 30 features
Updated data/test_features.csv: 30 features
